# 05a — Loss Functions

Corresponds to: Note 05 — Loss Functions

**Libraries introduced in this notebook:** No new libraries — numpy only (introduced in Note 03).

**What this notebook demonstrates:**
1. MSE — formula, gradient, disproportionate penalty on large errors
2. Binary cross-entropy — MLE derivation as code, formula, gradient
3. Categorical cross-entropy — one-hot encoding, reduction to −log(ŷ_c), gradient
4. MSE vs BCE saturation — gradient comparison on a confidently wrong prediction
5. Loss as a surface — loss as a function of one weight, minimum visible

---
## Part 1: Mean Squared Error

MSE measures the average squared distance between predictions and true values.

For a single sample: L = (y − ŷ)²  
For n samples: L = (1/n) Σ (yᵢ − ŷᵢ)²

Gradient: ∂L/∂ŷ = 2(ŷ − y)

We verify the worked example from the note and show that squaring penalises large errors disproportionately.

In [ ]:
import numpy as np

def mse(y, y_hat):
    return np.mean((y - y_hat) ** 2)

def mse_grad(y, y_hat):
    return 2 * (y_hat - y)

# Worked example from note: Model A vs Model B
y = np.array([3.0, 3.0, 3.0, 3.0])

y_hat_A = np.array([1.0, 1.0, 1.0, 1.0])  # errors: [2, 2, 2, 2]
y_hat_B = np.array([3.0, 3.0, 3.0, -5.0]) # errors: [0, 0, 0, 8]

print('MSE — Model A vs Model B:')
print(f'  Model A errors: {y - y_hat_A}')
print(f'  Model A MSE:    {mse(y, y_hat_A):.4f}')
print(f'  Model B errors: {y - y_hat_B}')
print(f'  Model B MSE:    {mse(y, y_hat_B):.4f}')
print()
print('Same total absolute error (8), but Model B MSE is 4x higher')
print('because squaring amplifies the single large error.')
print()

# Gradient at different error magnitudes
print('MSE gradient ∂L/∂ŷ = 2(ŷ − y) at y = 0, various ŷ:')
print(f'  {"ŷ":>6}   {"error (y−ŷ)":>12}   {"gradient":>10}')
print(f'  {"-"*6}   {"-"*12}   {"-"*10}')
for y_hat_val in [0.9, 0.5, 0.1, 0.01]:
    err = 1.0 - y_hat_val
    grad = mse_grad(1.0, y_hat_val)
    print(f'  {y_hat_val:>6.2f}   {err:>12.4f}   {grad:>10.4f}')

---
## Part 2: Binary Cross-Entropy

BCE is derived from maximum likelihood for a Bernoulli output.

The derivation has 5 steps:
1. Model output as probability: P(y=1|x) = ŷ, P(y=0|x) = 1 − ŷ
2. Combine both cases: P(y|x) = ŷʸ · (1−ŷ)^(1−y)
3. Likelihood over dataset: Π P(yᵢ|xᵢ)
4. Log-likelihood: Σ [yᵢ log(ŷᵢ) + (1−yᵢ) log(1−ŷᵢ)]
5. Negate to get loss: BCE = −(1/n) Σ [yᵢ log(ŷᵢ) + (1−yᵢ) log(1−ŷᵢ)]

When paired with sigmoid, the gradient simplifies to ∂L/∂z = ŷ − y.

We verify the table from the note and the gradient cancellation.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce(y, y_hat):
    # clip to avoid log(0)
    y_hat = np.clip(y_hat, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

# Step 2 — verify the combined probability formula
print('Step 2 — P(y|x) = ŷ^y · (1−ŷ)^(1−y):')
y_hat = 0.7
print(f'  y=1: {y_hat}^1 · {1-y_hat:.1f}^0 = {y_hat**1 * (1-y_hat)**0:.4f}  (should equal ŷ = {y_hat})')
print(f'  y=0: {y_hat}^0 · {1-y_hat:.1f}^1 = {y_hat**0 * (1-y_hat)**1:.4f}  (should equal 1−ŷ = {1-y_hat:.1f})')
print()

# Verify the loss table from the note
print('BCE loss table (single sample):')
print(f'  {"y":>4}   {"ŷ":>6}   {"Loss":>8}   Interpretation')
print(f'  {"-"*4}   {"-"*6}   {"-"*8}   {"-"*25}')
cases = [
    (1, 0.9, 'confident and correct'),
    (1, 0.5, 'uncertain'),
    (1, 0.1, 'confident and wrong'),
    (0, 0.1, 'confident and correct'),
    (0, 0.5, 'uncertain'),
    (0, 0.9, 'confident and wrong'),
]
for y_val, yh_val, label in cases:
    loss = bce(np.array([y_val]), np.array([yh_val]))
    print(f'  {y_val:>4}   {yh_val:>6.1f}   {loss:>8.3f}   {label}')

The gradient of BCE with sigmoid simplifies to ŷ − y. We verify this cancellation numerically — the chain rule result and the direct formula must agree.

In [ ]:
# Gradient cancellation: ∂L/∂z = ŷ − y
# Via chain rule: ∂L/∂ŷ × dσ/dz
# Direct formula: ŷ − y

print('BCE + sigmoid gradient cancellation:')
print(f'  {"y":>4}   {"ŷ":>6}   {"chain rule ∂L/∂z":>20}   {"direct ŷ−y":>12}   match')
print(f'  {"-"*4}   {"-"*6}   {"-"*20}   {"-"*12}   {"-"*5}')

test_cases = [(1, 0.9), (1, 0.5), (1, 0.1), (0, 0.1), (0, 0.9)]
for y_val, yh_val in test_cases:
    dL_dyhat = -y_val / yh_val + (1 - y_val) / (1 - yh_val)
    dsig     = yh_val * (1 - yh_val)
    chain    = dL_dyhat * dsig
    direct   = yh_val - y_val
    match    = 'yes' if abs(chain - direct) < 1e-9 else 'NO'
    print(f'  {y_val:>4}   {yh_val:>6.1f}   {chain:>20.6f}   {direct:>12.6f}   {match}')

---
## Part 3: Categorical Cross-Entropy

CCE extends BCE to K classes. The true label is a one-hot vector — a single 1 at the correct class, zeros elsewhere.

Formula: L = −Σₖ yₖ log(ŷₖ)

Because y is one-hot, this reduces to: L = −log(ŷ_c) where c is the correct class.

We verify the worked example from the note and show the gradient ∂L/∂zₖ = ŷₖ − yₖ.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())  # subtract max for numerical stability
    return e / e.sum()

def cce(y_onehot, y_hat):
    y_hat = np.clip(y_hat, 1e-12, 1.0)
    return -np.sum(y_onehot * np.log(y_hat))

# Worked example from note: true class = 1 (0-indexed), y_hat = [0.1, 0.7, 0.2]
y_onehot = np.array([0.0, 1.0, 0.0])
y_hat    = np.array([0.1, 0.7, 0.2])

loss_full   = cce(y_onehot, y_hat)
loss_direct = -np.log(y_hat[1])  # −log(ŷ_c)

print('CCE — worked example (true class = 1, ŷ = [0.1, 0.7, 0.2]):')
print(f'  Full sum:    −(0×log(0.1) + 1×log(0.7) + 0×log(0.2)) = {loss_full:.4f}')
print(f'  Direct:      −log(0.7) = {loss_direct:.4f}')
print(f'  Match: {abs(loss_full - loss_direct) < 1e-9}')
print()

# Loss table from note
print('CCE loss for different confidence levels (true class = 1):')
print(f'  {"ŷ":>22}   {"Loss":>8}   Interpretation')
print(f'  {"-"*22}   {"-"*8}   {"-"*22}')
scenarios = [
    (np.array([0.1, 0.7, 0.2]), 'confident and correct'),
    (np.array([0.3, 0.4, 0.3]), 'uncertain'),
    (np.array([0.05, 0.05, 0.9]), 'confident and wrong'),
]
for yh, label in scenarios:
    print(f'  {str(yh):>22}   {cce(y_onehot, yh):>8.3f}   {label}')
print()

# Gradient ∂L/∂zₖ = ŷₖ − yₖ
print('CCE + softmax gradient ∂L/∂zₖ = ŷₖ − yₖ:')
grad = y_hat - y_onehot
for k, (g, yh_k, y_k) in enumerate(zip(grad, y_hat, y_onehot)):
    direction = 'push up toward 1' if y_k == 1 else 'push down toward 0'
    print(f'  k={k}: ŷₖ − yₖ = {yh_k:.1f} − {y_k:.1f} = {g:+.1f}  ({direction})')

---
## Part 4: MSE vs BCE Saturation

When a sigmoid network makes a confidently wrong prediction, MSE suppresses the gradient because the sigmoid derivative is near zero. BCE cancels this suppression algebraically.

We reproduce the comparison from the note: y = 1, ŷ = 0.01.

In [ ]:
# Saturation comparison from note: y = 1, y_hat = 0.01
y_val  = 1.0
y_hat_val = 0.01

# MSE gradient through sigmoid
dL_dyhat_mse = 2 * (y_hat_val - y_val)
dsig         = y_hat_val * (1 - y_hat_val)
dL_dz_mse    = dL_dyhat_mse * dsig

# BCE gradient through sigmoid
dL_dz_bce = y_hat_val - y_val

print('Saturation comparison — y = 1, ŷ = 0.01 (confidently wrong):')
print()
print('MSE + sigmoid:')
print(f'  ∂L/∂ŷ  = 2(ŷ − y)          = 2({y_hat_val} − {y_val}) = {dL_dyhat_mse:.4f}')
print(f'  dσ/dz  = σ(z)(1 − σ(z))    = {y_hat_val} × {1-y_hat_val:.2f} = {dsig:.4f}')
print(f'  ∂L/∂z  = {dL_dyhat_mse:.4f} × {dsig:.4f}  = {dL_dz_mse:.4f}  ← suppressed')
print()
print('BCE + sigmoid:')
print(f'  ∂L/∂z  = ŷ − y             = {y_hat_val} − {y_val} = {dL_dz_bce:.4f}  ← strong signal')
print()

ratio = abs(dL_dz_bce) / abs(dL_dz_mse)
print(f'BCE gradient is {ratio:.0f}x larger than MSE gradient for this sample.')
print()

# Show across a range of confidently wrong predictions
print('Gradient comparison across confidence levels (y = 1):')
print(f'  {"ŷ":>6}   {"MSE ∂L/∂z":>12}   {"BCE ∂L/∂z":>12}   ratio')
print(f'  {"-"*6}   {"-"*12}   {"-"*12}   {"-"*8}')
for yh in [0.9, 0.5, 0.1, 0.01, 0.001]:
    g_mse = 2 * (yh - 1.0) * yh * (1 - yh)
    g_bce = yh - 1.0
    r = abs(g_bce) / abs(g_mse) if g_mse != 0 else float('inf')
    print(f'  {yh:>6.3f}   {g_mse:>12.6f}   {g_bce:>12.6f}   {r:>6.1f}x')

---
## Part 5: Loss as a Surface

The loss function defines a surface over parameter space. We make this concrete by computing MSE as a function of a single weight w, holding everything else fixed.

This shows why gradient descent works — the surface has a minimum, and the gradient always points toward it.

In [ ]:
# Single-weight loss surface: y = w * x, one sample x=1, y=2
# L(w) = (y - w*x)^2 = (2 - w)^2  → minimum at w = 2
x_val = 1.0
y_val = 2.0

w_values = np.linspace(-1, 5, 13)
L_values = (y_val - w_values * x_val) ** 2
grad_values = 2 * (w_values * x_val - y_val) * x_val  # ∂L/∂w

print('Loss surface L(w) = (2 − w)² — minimum at w = 2:')
print(f'  {"w":>6}   {"L(w)":>8}   {"∂L/∂w":>10}   update direction')
print(f'  {"-"*6}   {"-"*8}   {"-"*10}   {"-"*16}')
for w, L, g in zip(w_values, L_values, grad_values):
    if abs(g) < 1e-9:
        direction = 'minimum — no update'
    elif g > 0:
        direction = 'decrease w'
    else:
        direction = 'increase w'
    print(f'  {w:>6.2f}   {L:>8.4f}   {g:>10.4f}   {direction}')
print()
print(f'Minimum loss = {L_values.min():.4f} at w = {w_values[L_values.argmin()]:.2f}')
print('Gradient is zero at the minimum — gradient descent stops here.')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| MSE large-error penalty | Model B MSE = 16 vs Model A MSE = 4 — same total error, 4× higher cost due to one large error |
| MSE gradient | ∂L/∂ŷ = 2(ŷ − y) — linear in error, well-behaved for regression |
| BCE derivation | P(y\|x) = ŷʸ·(1−ŷ)^(1−y) — both cases in one formula; log converts product to sum |
| BCE gradient cancellation | Chain rule gives ŷ − y exactly — sigmoid saturation cancels algebraically |
| CCE reduction | L = −Σ yₖ log(ŷₖ) reduces to −log(ŷ_c) — only correct class probability matters |
| CCE gradient | ∂L/∂zₖ = ŷₖ − yₖ — same clean form as BCE; softmax Jacobian cancels |
| MSE vs BCE saturation | At ŷ = 0.01, y = 1: BCE gradient ≈ 50× larger than MSE gradient |
| Loss surface | L(w) is a parabola — gradient points toward minimum, zero at minimum |